# Codec restoration benchmark

Set `SOURCE` below and Run All. `scripts/run_xp.py` must have filled
`artifacts/xp/` first (the model renders come from a GPU pod).

Four methods against the clean original of one chunk, at three MP3 bitrates:
the SAME round-trip (S and L), Apollo, and A2SB. `input` is the compressed MP3 scored against the master — the damage
itself, and the do-nothing floor every method must beat.

BSS-SDR is the published norm: a 512-tap filter of the master is fitted
first, so gain, EQ and small delays are forgiven. Plain SDR charges any
waveform difference; SI-SNR forgives gain only. Spectral SNR compares
STFT magnitudes, so phase costs nothing; LSD (lower is better) is the
gap between the two log-power spectrograms — the closest number to
comparing spectrograms by eye, and what bandwidth-extension papers
publish. The SAME decoder re-realises phase, which both metrics punish however it
sounds — so read the table together with the spectrograms and the players.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from grooveback import audio as ga

SOURCE = "aerofunk"  # "aerofunk" | "codec" — set, then Run All

XP = Path("../artifacts/xp")
BITRATES = ("64k", "128k", "192k")
ORDER = ("input", "same-s", "same-l", "apollo", "a2sb")

RESULTS = json.loads((XP / "results.json").read_text())[SOURCE]

In [ ]:
print(f"{SOURCE} — BSS-SDR / SDR / SI-SNR / SpecSNR / LSD(lower better), dB vs master")
for bitrate in BITRATES:
    print(f"\n{bitrate}")
    for method in ORDER:
        scores = RESULTS[bitrate].get(method)
        if scores:
            print(f"  {method:8}{scores['bss_sdr_db']:>7.1f} /{scores['sdr_db']:>6.1f} /"
                  f"{scores['si_snr_db']:>6.1f} /{scores['spectral_snr_db']:>6.1f} /"
                  f"{scores['lsd_db']:>6.1f}")

## The fill band on its own

The band above the measured codec edge, against the master. Three views:
waveform SDR (phase-bound; silence = 0), spectral SNR (phase-blind; silence
= 0), and log-spectral distance (lower is better — the RMS gap between the
two log-power spectrograms, the closest number to comparing spectrogram
colours by eye, and what bandwidth-extension papers publish).

In [ ]:
print(f"{SOURCE} — fill band: waveform SDR / spectral SNR / LSD (silence = 0 / 0 / max)")
for bitrate in BITRATES:
    edge = RESULTS[bitrate]["edge_hz"] / 1000
    print(f"\n{bitrate} @ {edge:.1f}k")
    for method in ORDER:
        scores = RESULTS[bitrate].get(method)
        if scores:
            print(f"  {method:8}{scores['fill_sdr_db']:>7.1f} /"
                  f"{scores['fill_spectral_snr_db']:>6.1f} /"
                  f"{scores['fill_lsd_db']:>6.1f}")

## Spectrograms

In [ ]:
def spectrogram(ax, wav, title):
    audio, sr = ga.load(wav)
    ax.imshow(ga.spectrogram_db(audio), origin="lower", aspect="auto",
              cmap="magma", vmin=-100, vmax=0,
              extent=[0, audio.shape[1] / sr, 0, sr / 2 / 1000])
    ax.set_title(title, fontsize=9)

for bitrate in BITRATES:
    wavs = [("original", XP / SOURCE / "original.wav")] + [
        (method, XP / SOURCE / bitrate / f"{method}.wav") for method in ORDER]
    wavs = [(title, path) for title, path in wavs if path.exists()]
    fig, axes = plt.subplots(2, 3, figsize=(13, 6), constrained_layout=True,
                             sharex=True, sharey=True)
    for ax, (title, path) in zip(axes.flat, wavs):
        spectrogram(ax, path, title)
    for ax in axes.flat[len(wavs):]:
        ax.axis("off")
    fig.suptitle(f"{SOURCE} @ {bitrate}, dBFS, kHz over seconds")
    plt.show()

## Listen

Level-matched to −14 LUFS with one shared headroom gain per set.

In [ ]:
for bitrate in BITRATES:
    print(f"── {bitrate} " + "─" * 40)
    for name in ("original", *ORDER):
        wav = XP / SOURCE / bitrate / "listen" / f"{name}.wav"
        if wav.exists():
            print(name)
            display(Audio(str(wav)))